In [ ]:
'''
-------------------
FEATURE ENGINEERING
-------------------
'''
%cd ~/workspace/FINRL
from lib.utils.ingest import pipeline_ingest_data
from lib.utils.plotting import *
from lib.utils.technical import *
from lib.utils.utils import *
from typing import List
import pandas as pd
import polars as pl
import yfinance as yf

START_DATE = "1995-01-01"
END_DATE = "2005-06-01"

all_tickers = ["SPY"]

print(f"Ingesting {len(all_tickers)} tickers...")
df = pipeline_ingest_data(
    tickers=all_tickers,
    start_date=START_DATE,
    end_date=END_DATE
)

weekly_df = (
    df
    .sort(["Ticker", "Date"])
    .group_by_dynamic(
        "Date",
        every="1w",
        group_by="Ticker",
    )
    .agg(
        [
            pl.col("Open").first().alias("Open"),
            pl.col("High").max().alias("High"),
            pl.col("Low").min().alias("Low"),
            pl.col("Close").last().alias("Close"),
            pl.col("Volume").sum().alias("Volume"),
        ]
    )
)

In [ ]:
SHORT_WINDOW = 20
LONG_WINDOW = 120
PRIMARY_TICKER = "SPY"

In [ ]:
from lib.utils.utils import utils_standardize_rolling
from lib.utils.technical import append_weekly_macd_klinger_hist
window_size = SHORT_WINDOW

features_dict = {
    "RSI_20": 126,
    "MACD_Signal": 126,
    "Klinger_Signal": 126,
    "W_MACD_Signal": 126,
    "W_Klinger_Signal": 126,
    "Log_Return": 126,
    f"Amihud_{SHORT_WINDOW}d_{LONG_WINDOW}d_Ratio": 252,
    "W_Klinger_Hist" : 60,
    "W_MACD_Hist": 60,
    "MACD_Hist": 30,
    "Klinger_Hist": 30
}


df = append_panel_rolling_amihud(df, window_size)
df = append_panel_rolling_amihud_ratio(
    df,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW
)

df = append_log_returns(df)

df = utils_add_klinger_and_macd_signals(df)
df = append_rsi(df)
df = append_weekly_macd_klinger_hist(df, weekly_df)

df = df.with_columns(
    (
        1 / ((pl.col("MACD_Signal") ** 2)
        + (pl.col("Klinger_Signal") ** 2) + 1e-5).sqrt()
    ).alias("Compression")
)

features_dict["Compression"] = 252

display(df.head())